<a href="https://colab.research.google.com/github/SergiSama/UIC-CRB1-2026-2027/blob/main/m1_python_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 1 · Notebook 1 — Python for Bioengineers
### Computing, Robotics & Bionics · companion to A. Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch*

Géron's book assumes you can already read and write idiomatic Python, but ships **no** Python-basics
notebook. This **self-contained** notebook fills that gap — the exact subset of Python you need to read
Scikit-Learn and PyTorch code — with biomedical examples throughout. Run it in **Google Colab**
(*Runtime → Run all*, or **Shift+Enter** per cell). Nothing to install.

**Contents**
1. Numbers, strings, booleans, `None`
2. Names, objects & the aliasing trap
3. Containers: list, tuple, dict, set
4. Indexing & slicing
5. Control flow & truthiness
6. Comprehensions & generators
7. Functions
8. Errors & exceptions
9. Classes & objects (just enough to read ML APIs)
10. Modules & the scientific stack
11. Mini-capstone: a tiny patient registry in pure Python

Sections end with an **Exercise**; run the **Solution** cell to check yourself.

---
## 1 · Numbers, strings, booleans, `None`

The atoms of Python. Note integer vs float division, f-strings for formatting, and that `None` is
Python's "no value".

In [ ]:
age = 58                 # int
glucose = 131.4          # float
name = "patient-017"     # str
is_diabetic = glucose >= 126   # bool (a comparison)
missing = None           # absence of a value

print(type(age), type(glucose), type(name), type(is_diabetic))
print(7 / 2, 7 // 2, 7 % 2, 2 ** 10)     # true div, floor div, modulo, power
print(f"{name}: age {age}, glucose {glucose:.1f} -> diabetic? {is_diabetic}")  # f-string

In [ ]:
# Python also has a built-in complex type -- rare in everyday code, but the natural type for
# frequency-domain work (e.g. the Fourier transform of an ECG signal, later in the course)
z1 = 1 + 5j
z2 = complex(2, 3)
print(type(z1), z1, "| real:", z1.real, "| imag:", z1.imag)
print("z1 + z2 =", z1 + z2, "| z1 * z2 =", z1 * z2)

In [ ]:
# Strings are sequences: index, slice, and call methods
s = "BRCA1"
print(s[0], s[-1], s[1:4], len(s), s.lower(), "BRCA" in s)

In [ ]:
# Triple quotes span multiple lines -- handy for a report template or a docstring
report = """Patient: patient-017
Glucose: 131.4 mg/dL
Diagnosis: pending"""
print(report)

# The same text written with escape characters instead: \n means "new line", \t means "tab"
print("Patient: patient-017\nGlucose: 131.4 mg/dL\tDiagnosis: pending")

---
## 2 · Names, objects & the aliasing trap

In Python a variable is a **name** bound to an object — a luggage tag, not a box. Assignment never
copies. Two names can tag the **same** mutable object, so changing one changes "both". This is the
single most common silent bug in data code.

In [ ]:
a = [1, 2, 3]
b = a              # b tags the SAME list (no copy!)
b.append(4)
print("a is now:", a)        # a changed too -> aliasing

c = a.copy()       # explicit copy
c.append(99)
print("a unchanged by c:", a)

**Immutable** objects (`int`, `float`, `str`, `tuple`, `bool`) cannot change in place, so they are
never aliased in a surprising way. **Mutable** objects (`list`, `dict`, `set`, and later NumPy arrays and
Pandas frames) can be — copy on purpose when you need independence.

---
## 3 · Containers: list, tuple, dict, set

Four containers carry almost all data before it becomes a NumPy array.

In [ ]:
features = [12.4, 0.27, 84.1]          # list: ordered, mutable, grows
shape    = (569, 30)                   # tuple: fixed record (e.g. an array shape)
patient  = {"id": 17, "age": 58,       # dict: key -> value (a record)
            "diagnosis": "benign"}
genes    = {"BRCA1", "BRCA2", "TP53"}  # set: unique, unordered

features.append(1.5)                    # lists grow
print("list:", features)
print("dict lookup:", patient["age"], "| keys:", list(patient.keys()))
print("set membership:", "TP53" in genes, "| de-duplicated:", {1,1,2,2,3})

**Exercise 3.** Build a dict `vitals` with keys `"hr"`, `"sbp"`, `"temp"` and any values. Then (a)
print just the value for `"sbp"`, and (b) add a new key `"spo2"` with value 98.

In [ ]:
# Solution 3
vitals = {"hr": 72, "sbp": 128, "temp": 36.8}
print("(a)", vitals["sbp"])
vitals["spo2"] = 98
print("(b)", vitals)

**Why tuples (not lists) can be dictionary keys.** A dict key must be *hashable* -- Python
needs to compute the same hash for it every time. Immutable objects (`tuple`, `str`, `int`)
qualify; mutable ones (`list`, `dict`, `set`) don't, because their contents -- and therefore
their hash -- could change after being used as a key. Same mutability idea as Section 2, now
showing up as a hard rule instead of a silent bug.

In [ ]:
locus = (17, 41196312)          # (chromosome, position) -- a fixed record, so it's hashable
variant_notes = {locus: "pathogenic"}
print(variant_notes[locus])

try:
    bad_key = [17, 41196312]    # a list is mutable -> unhashable
    variant_notes[bad_key] = "oops"
except TypeError as e:
    print("Can't use a list as a key:", e)

---
## 4 · Indexing & slicing

Sequences index from `0`; negative indices count from the end. Slicing is `seq[start:stop:step]` with
`stop` **excluded**. The *same* syntax, generalised to many dimensions, returns in NumPy and Pandas — so
fluency here pays off everywhere.

In [ ]:
x = list(range(10))      # [0, 1, ..., 9]
print(x[0], x[-1])       # first, last
print(x[2:5])            # [2, 3, 4]   (stop excluded)
print(x[:3], x[7:])      # first three, last three
print(x[::2])            # every second element
print(x[::-1])           # reversed

In [ ]:
# Indexing also lets you mutate a list in place -- it's an assignment target, not just a read
x[0] = -1
print("after x[0] = -1:", x)

# An index past the end raises IndexError rather than silently returning something:
try:
    print(x[100])
except IndexError as e:
    print("IndexError:", e)

---
## 5 · Control flow & truthiness

`if`/`elif`/`else`, `for`, `while`. Loop directly over items (not indices); use `enumerate` for an index
and `zip` to walk two sequences together. Empty containers, `0`, `""` and `None` are all **falsy**.

In [ ]:
glucoses = [88, 145, 99, 210, 126]
for i, g in enumerate(glucoses):
    label = "high" if g >= 126 else "ok"        # ternary expression
    print(f"sample {i}: {g} -> {label}")

names = ["A", "B", "C"]
for nm, g in zip(names, glucoses):              # walk two lists in step
    print(nm, g)

print("truthiness:", bool([]), bool([0]), bool(""), bool("x"), bool(None))

---
## 6 · Comprehensions & generators

The Pythonic way to build a sequence from another — you will read these constantly in ML code. A
**generator** (round brackets) is the lazy version that yields items one at a time (memory-friendly).

In [ ]:
squares = [g**2 for g in glucoses]                    # list comprehension
high     = [g for g in glucoses if g >= 126]          # with a filter
labels   = ["high" if g >= 126 else "ok" for g in glucoses]
name2id  = {nm: i for i, nm in enumerate(names)}      # dict comprehension
print(squares); print(high); print(labels); print(name2id)

total = sum(g for g in glucoses if g >= 100)          # generator (no list built)
print("sum of glucoses >= 100:", total)

**Exercise 6.** From `glucoses`, build (a) a list of only the values below 100, and (b) a dict mapping
each sample index to `True`/`False` for "is high" (>= 126).

In [ ]:
# Solution 6
print("(a)", [g for g in glucoses if g < 100])
print("(b)", {i: (g >= 126) for i, g in enumerate(glucoses)})

---
## 7 · Functions

Functions package logic. They take positional and keyword arguments, can have **defaults**, return
values, and are themselves **objects** (you can pass them around) — which is why Scikit-Learn and PyTorch
accept a function as an argument (a metric, an activation, a transform). Type hints document intent.

In [ ]:
def standardize(x: float, mean: float = 0.0, std: float = 1.0) -> float:
    '''Return the z-score (x - mean) / std.'''   # docstring
    return (x - mean) / std

print(standardize(140, mean=120, std=15))     # 1.33...
f = standardize                                # functions are objects
print(f(140, 120, 15))

relu = lambda z: max(0.0, z)                   # tiny anonymous function
print("relu:", relu(-3), relu(2))

def summary(values):                           # multiple returns via a tuple
    return min(values), max(values), sum(values)/len(values)
lo, hi, avg = summary(glucoses)
print("lo/hi/avg:", lo, hi, round(avg, 1))

---
## 8 · Errors & exceptions

When something goes wrong Python **raises an exception**. Catch the ones you expect with `try`/`except`
so a single bad record doesn't crash a whole pipeline.

In [ ]:
def safe_ratio(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return float("nan")        # not-a-number, a common 'missing' marker

print(safe_ratio(10, 2), safe_ratio(10, 0))

readings = ["72", "80", "n/a", "75"]
clean = []
for r in readings:
    try:
        clean.append(int(r))
    except ValueError:
        pass                       # skip unparseable entries
print("parsed:", clean)

In [ ]:
# The same try/except pattern guards against a missing dict key -- common when a field
# wasn't recorded for a given patient
patient = {"id": 17, "age": 58}
for field in ["age", "glucose"]:
    try:
        print(field, "=", patient[field])
    except KeyError:
        print(field, "not recorded for this patient")

---
## 9 · Classes & objects (just enough to read ML APIs)

You will **read** far more classes than you write. Scikit-Learn and PyTorch are object-oriented: a model
is an **object** with **attributes** (data it stores) and **methods** (things it does). The pattern you
will see everywhere is `model.fit(X, y)` then `model.predict(X)`, with learned attributes ending in `_`
(e.g. `model.coef_`). Here is a tiny version of that pattern.

In [ ]:
class MinMaxScaler1D:
    '''A toy of sklearn's scaler: note the fit / transform pattern.'''
    def fit(self, data):
        self.lo_ = min(data)              # learned attributes end in _
        self.hi_ = max(data)
        return self                       # return self so calls can chain
    def transform(self, data):
        return [(v - self.lo_) / (self.hi_ - self.lo_) for v in data]
    def fit_transform(self, data):
        return self.fit(data).transform(data)

scaler = MinMaxScaler1D()
scaled = scaler.fit_transform([60, 72, 80])
print("attributes:", scaler.lo_, scaler.hi_)     # object.attribute
print("method result:", scaled)                  # object.method(args)

**Exercise 9.** Add a method `inverse_transform(self, scaled)` to `MinMaxScaler1D` that maps a scaled
list back to original units. Verify it round-trips `[60, 72, 80]`.

In [ ]:
# Solution 9
class MinMaxScaler1D:
    def fit(self, data):
        self.lo_, self.hi_ = min(data), max(data); return self
    def transform(self, data):
        return [(v - self.lo_) / (self.hi_ - self.lo_) for v in data]
    def fit_transform(self, data):
        return self.fit(data).transform(data)
    def inverse_transform(self, scaled):
        return [s * (self.hi_ - self.lo_) + self.lo_ for s in scaled]

sc = MinMaxScaler1D().fit([60, 72, 80])
print(sc.inverse_transform(sc.transform([60, 72, 80])))   # -> [60, 72, 80]

---
## 10 · Modules & the scientific stack

Code is organised into **modules** you `import`. The scientific stack you will use all course is imported
with conventional short names — memorise these four lines; they head almost every notebook.

In [ ]:
import numpy as np            # arrays (Notebook 2)
import pandas as pd           # tables  (Notebook 3)
import matplotlib.pyplot as plt   # plots (Notebook 4)
from math import pi, sqrt     # import specific names from a module
print("stack ready:", np.__version__, pd.__version__)
print("pi =", round(pi, 4), "| sqrt(2) =", round(sqrt(2), 4))

---
## 11 · Mini-capstone — a tiny patient registry in pure Python

Combine the whole notebook: a list of patient dicts, processed with loops, comprehensions, functions and
exceptions — *no NumPy or Pandas yet*. (In Notebooks 2–3 you will see how much shorter and faster this
becomes once the data is an array / DataFrame — that contrast is the point.)

In [ ]:
registry = [
    {"id": 1, "age": 58, "glucose": 131.4, "diagnosis": "benign"},
    {"id": 2, "age": 71, "glucose": 205.0, "diagnosis": "malignant"},
    {"id": 3, "age": 19, "glucose": 88.0,  "diagnosis": "benign"},
    {"id": 4, "age": 65, "glucose": 150.0, "diagnosis": "malignant"},
    {"id": 5, "age": 40, "glucose": None,  "diagnosis": "benign"},   # missing value
]

# Mean glucose, skipping missing values:
vals = [p["glucose"] for p in registry if p["glucose"] is not None]
mean_glucose = sum(vals) / len(vals)
print("mean glucose (non-missing):", round(mean_glucose, 1))

# Count per diagnosis (a hand-rolled group-by):
counts = {}
for p in registry:
    counts[p["diagnosis"]] = counts.get(p["diagnosis"], 0) + 1
print("counts by diagnosis:", counts)

# Flag high glucose with a helper function:
def flag(p):
    g = p["glucose"]
    return "unknown" if g is None else ("high" if g >= 126 else "ok")
print("flags:", [(p["id"], flag(p)) for p in registry])

---
### You now read Python
Data types, names and aliasing, the four containers, slicing, control flow, comprehensions, functions,
exceptions, and the object pattern behind every ML API. **Next:** Notebook 2 (NumPy) replaces these
Python loops with fast whole-array operations — re-read the capstone afterwards and notice how it
collapses to a few array expressions.